## Namespaces and Scope:
- It follows LEGB rule:
- L: Local 
- E: Enclosing
- G: Global
- B: Built-ins

In [8]:
# Local Scope
def run_agent():
    agent_id = 'Agent_Alpha'
    print(f"Inside Function: {agent_id}")

run_agent()
try: 
    print(agent_id)     #bcs accessing outside of function 
except NameError as e:
    print(f"Error caught: {e}")



Inside Function: Agent_Alpha
Error caught: name 'agent_id' is not defined


In [9]:
# Enclosing Scope 
def parent_pipeline():
    rag_context = 'RAG retrieved completed'
    def child_llm():
        print(f"LLM response: {rag_context}")
    child_llm()

parent_pipeline()


LLM response: RAG retrieved completed


In [10]:
GLOBAL_MODEL = 'gpt-4o'

def query_one():
    print(f"Query one answer: {GLOBAL_MODEL}")

def query_two():
    print(f"Query two answer: {GLOBAL_MODEL}")

query_one()
query_two()

Query one answer: gpt-4o
Query two answer: gpt-4o


In [15]:
# UnboundLocalError mistakes

total_token_used = 100
def cost_token():
    total_token_used = total_token_used + 50

try:
    cost_token()
except UnboundLocalError as err:
    print(f'Production Bug: {err}')

Production Bug: cannot access local variable 'total_token_used' where it is not associated with a value


## nonlocal and global keywords used:

In [20]:
# Using of global keyword: point to be noted, using of global keyword very baad idea in python

user_active = 0
def connected_user():
    global user_active
    user_active +=1

connected_user()
connected_user()
connected_user()

print(f"Total active users: {user_active}")

Total active users: 3


In [21]:
# Using nonlocal keyword to  change or update the variable in scope of Enclosing variable

def create_token_cost():
    session_cost = 0.0
    def add_cost(query_cost):
        nonlocal session_cost
        session_cost += query_cost
        return session_cost
    return add_cost
tracker = create_token_cost()
print(f"Run 1 cost ${tracker(0.2)}")
print(f"Run 2 cost ${tracker(0.5)}")

Run 1 cost $0.2
Run 2 cost $0.7


In [23]:
# Namespace isolation: wrong way to define global variables
current_user_memory = []
def process_chat(user_msg):
    global current_user_memory
    current_user_memory.append(user_msg)
    return f"Context size {len(current_user_memory)} messags."

process_chat('This is my secret passwords @124')
process_chat('Hello ! My name is Majid')

# Now the problem is that dono users ki same dictionary mein chat save ho gi, which is very bad

'Context size 2 messags.'

In [24]:
# Right way to isolate namespaces\
def create_chat_session():
    current_user_memory = []
    def chat(msg):
        current_user_memory.append(msg)
        return f"Isolated chat history length: {len(current_user_memory)} messages"
    return chat

session_ali = create_chat_session()
session_ahmed = create_chat_session()

session_ali('Ali: Secret Data')
print("Ahmed session", session_ahmed("hello, Ahmed"))

Ahmed session Isolated chat history length: 1 messages


## Decoratos:

In [35]:
from functools import wraps

def my_logger(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Logging before call....")

        return func(*args, **kwargs)
    return wrapper
@my_logger
def generate_summery(text):
    "Summerize input docs using an LLM"
    clean_text = text.strip().lower()
    return clean_text

cleaned_output = generate_summery(' WhAt is Docs SuMmary   ')
print(f"Cleaned docs: {cleaned_output}")
print(f"Function name: {generate_summery.__name__}")
print(f"Function docs: {generate_summery.__doc__}")

Logging before call....
Cleaned docs: what is docs summary
Function name: generate_summery
Function docs: Summerize input docs using an LLM


In [43]:
# Example2: input validation or prompt injection guard
from functools import wraps

def guard_empty_prompts(func):
    @wraps(func)
    def wrappers(*args, **kwargs):
        prompt = args[0] if args else kwargs.get('prompt', "")
        if not prompt or prompt.strip() == "":
            raise ValueError(f'[Security issue] Prompt cannot be empty')
        return func(*args, **kwargs)
    return wrappers

@guard_empty_prompts
def ask_cluade(prompt: str) -> str:
    return f"Claude responding of: {prompt}"

print(ask_cluade("what is RAG pipeline in production"))
print(ask_cluade('hum'))

Claude responding of: what is RAG pipeline in production
Claude responding of: hum


In [52]:
# latency counting time datalog
import time
from functools import wraps

def track_execution_time(func):
    @wraps(func)
    def wrappers(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[Telemetry Function] {func.__name__} took {elapsed:.2} seconds")
        return result
    return wrappers

@track_execution_time
def vector_search(query):
    time.sleep(0.15)
    return ['RAG_01.docs', 'RAG_02.docs']

@track_execution_time
def generate_summarization(docs):
    time.sleep(3)
    return ['company_policy.docs', 'leave_company_policy.docs']

print(vector_search('What is MCP'))
print(generate_summarization('Agent RAG system'))

[Telemetry Function] vector_search took 0.15 seconds
['RAG_01.docs', 'RAG_02.docs']
[Telemetry Function] generate_summarization took 3.0 seconds
['company_policy.docs', 'leave_company_policy.docs']


In [61]:
# sanity check decorator that check whether user input is string or int?
from functools import wraps

def sanity_check(func):
    @wraps(func)
    def wrappers(*args, **kwargs):
        for arg in args:
            if not isinstance(arg, int):
                raise TypeError("[Security Guard Alert] system is only acccept integer as input")
        return func(*args, **kwargs)
    return wrappers

@sanity_check
def calc_sum(a,b):
    return a+b

# wrong input as string
@sanity_check
def my_name(text):
    return text

print(f"Calculation when enter Integer: {calc_sum(10,20)}")
print(my_name("Majid Hussain"))


Calculation when enter Integer: 30


TypeError: [Security Guard Alert] system is only acccept integer as input

In [66]:
# Taking input as in factory layer with three 

from functools import wraps
def with_prex(tag: str):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            print(f"[{tag}] function {func.__name__} is started.")
            result = func(*args, **kwargs)
            print(f"[{tag}] function is ended !.")
            return result
        return wrapper
    return decorator

@with_prex('AI System')
def greet_name(name):
    print(f" Hello , {name}")

@with_prex('DataBase')
def fetch_record():
    print('Fetching 5 users record from disk') 


greet_name('Majid')
print()
fetch_record()

[AI System] function greet_name is started.
 Hello , Majid
[AI System] function is ended !.

[DataBase] function fetch_record is started.
Fetching 5 users record from disk
[DataBase] function is ended !.


In [70]:
# Example2: security guard for checking model temperature if it too high capping within given
from functools import wraps
def enforce_max_temp(max_temp_allowed):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            temp = kwargs.get('temperature' , 0.7)
            if temp > max_temp_allowed:
                print(f"[Security] temp {temp} is too high capping to {max_temp_allowed}")
                kwargs['temperature'] = max_temp_allowed
            return func(*args, **kwargs)
        return wrapper
    return decorator
@enforce_max_temp(max_temp_allowed=0.5)
def call_chat_model(prompt , temperature = 0.7):
    return f"Model output for {prompt} with temp = {temperature}"

print(call_chat_model('Write a formal email', temperature= 0.9))
print()
print(call_chat_model('write code', temperature=0.2))


[Security] temp 0.9 is too high capping to 0.5
Model output for Write a formal email with temp = 0.5

Model output for write code with temp = 0.2
